# Batch Spectral Test Evaluation for 50+ LCGs
### Automated computation of Tezuka-normalized spectral scores across a large set of LCG configurations.

## This notebook evaluates a broad collection of Linear Congruential Generators (LCGs) using the spectral test.  
## It applies the same geometry-of-numbers pipeline used in the main implementation,
## lattice construction, LLL reduction, shortest-vector extraction, and Tezuka normalization.
## However, it should be noted it performs it automatically across many generators.

# The goal is to compare spectral quality across a wide parameter space and identify good and bad LCGs.

import numpy as np
from sympy import Matrix
import pandas as pd


# ---------------------------------------------------------
# 1. Custom LLL reduction (integer-preserving)
# ---------------------------------------------------------

def lll_reduce(B, delta=0.75):
    B = Matrix(B)
    n = B.cols

    def gs(B):
        m = B.rows
        U = [B[:, i] for i in range(n)]
        U_star = [U[0]]
        mu = [[0]*n for _ in range(n)]
        mu[0][0] = 1

        for i in range(1, n):
            proj = Matrix.zeros(m, 1)
            for j in range(i):
                proj += (U[i].dot(U_star[j]) / U_star[j].dot(U_star[j])) * U_star[j]
            U_star.append(U[i] - proj)

            for j in range(i):
                mu[i][j] = U[i].dot(U_star[j]) / U_star[j].dot(U_star[j])
            mu[i][i] = 1

        return U_star, mu

    U_star, mu = gs(B)
    k = 1

    while k < n:
        # Size reduction
        for j in range(k-1, -1, -1):
            q = round(mu[k][j])
            if q != 0:
                B[:, k] -= q * B[:, j]

        U_star, mu = gs(B)

        # Lovász condition
        if U_star[k].dot(U_star[k]) >= (delta - mu[k][k-1]**2) * U_star[k-1].dot(U_star[k-1]):
            k += 1
        else:
            B.col_swap(k, k-1)
            U_star, mu = gs(B)
            k = max(k-1, 1)

    return B

# ---------------------------------------------------------
# 2. Shortest vector finder (column-based)
# ---------------------------------------------------------

def find_shortest_vector(B):
    m, n = B.shape
    shortest = None
    shortest_norm = float('inf')

    for i in range(n):
        v = B[:, i]
        norm = float(v.norm())
        if norm < shortest_norm:
            shortest_norm = norm
            shortest = v

    return shortest

# ---------------------------------------------------------
# 3. Dual lattice basis for LCG (Knuth/Tezuka)
# ---------------------------------------------------------

def build_lcg_dual_basis(a, m, s):
    B = []
    for i in range(s):
        row = [0] * s
        if i == 0:
            row[0] = m
        else:
            row[0] = pow(int(a), i, int(m))
        row[i] = 1
        B.append(row)
    return B

# ---------------------------------------------------------
# 4. Hermite constants γ_s for Tezuka normalization
# ---------------------------------------------------------

HERMITE_CONSTANTS = {
    2: 2 / np.sqrt(3),
    3: (2/np.sqrt(3))**(3/2),
    4: 2.0,
    5: 2.0**(5/4),
    6: 2.0**(3/2),
    7: 2.0**(7/4)
}

# ---------------------------------------------------------
# 5. Tezuka-normalized spectral score
# ---------------------------------------------------------

def tezuka_spectral_score(shortest_vector, m, s):
    v = np.array(shortest_vector, dtype=float).flatten()
    d_s = np.linalg.norm(v)

    gamma_s = HERMITE_CONSTANTS[s]
    best_possible = np.sqrt(gamma_s) * (m ** (1.0 / s))

    return (1.0 / d_s) / best_possible

LCG_DATABASE = [

    # ---------------------------------------------------------
    # 1. CLASSIC / HISTORICAL LCGs
    # ---------------------------------------------------------
    {"name": "Park–Miller (Minimal Standard)", "a": 16807, "m": 2**31 - 1},
    {"name": "RANDU (Infamous Bad LCG)", "a": 65539, "m": 2**31},
    {"name": "IBM RANDU Variant", "a": 65539, "m": 2**31},
    {"name": "Borland C++", "a": 22695477, "m": 2**32},
    {"name": "Turbo Pascal", "a": 134775813, "m": 2**32},
    {"name": "MSVC LCG", "a": 214013, "m": 2**32},
    {"name": "glibc rand()", "a": 1103515245, "m": 2**31},
    {"name": "Numerical Recipes LCG", "a": 1664525, "m": 2**32},
    {"name": "ANSI C LCG", "a": 1103515245, "m": 2**31},
    {"name": "UNIX Seventh Edition LCG", "a": 1103515245, "m": 2**31},

    # ---------------------------------------------------------
    # 2. MODERN / HIGH-QUALITY LCGs (L’Ecuyer-style)
    # ---------------------------------------------------------
    {"name": "L'Ecuyer (m = 2^31 - 1)", "a": 950706376, "m": 2**31 - 1},
    {"name": "L'Ecuyer (m = 2^32 - 5)", "a": 1588635695, "m": 2**32 - 5},
    {"name": "L'Ecuyer (m = 2^24 - 3)", "a": 6423135, "m": 2**24 - 3},
    {"name": "Knuth MMIX", "a": 6364136223846793005, "m": 2**64},
    {"name": "Java Util.Random", "a": 25214903917, "m": 2**48},
    {"name": "PCG Reference LCG", "a": 6364136223846793005, "m": 2**64},
    {"name": "PCG Alternate LCG", "a": 1442695040888963407, "m": 2**64},
    {"name": "PCG Experimental LCG", "a": 1181783497276652981, "m": 2**64},

    # ---------------------------------------------------------
    # 3. OBSCURE / LESS COMMON LCGs
    # ---------------------------------------------------------
    {"name": "Fishman LCG (m = 2^31 - 1)", "a": 950706376, "m": 2**31 - 1},
    {"name": "Lewis-Goodman-Miller LCG", "a": 16807, "m": 2**31 - 1},
    {"name": "Prime Modulus LCG (m = 2147483629)", "a": 40014, "m": 2147483629},
    {"name": "Prime Modulus LCG (m = 2147483647)", "a": 16807, "m": 2147483647},
    {"name": "Obscure LCG (m = 2^30 + 3)", "a": 12345, "m": 2**30 + 3},
    {"name": "Obscure LCG (m = 2^28 + 15)", "a": 54321, "m": 2**28 + 15},
    {"name": "Obscure LCG (m = 2^27 + 7)", "a": 98765, "m": 2**27 + 7},
    {"name": "Obscure LCG (m = 2^26 + 17)", "a": 192837, "m": 2**26 + 17},

    # ---------------------------------------------------------
    # 4. WEIRD MODULUS FAMILIES (2^k - c)
    # ---------------------------------------------------------
    {"name": "LCG m = 2^20 - 3", "a": 380985, "m": 2**20 - 3},
    {"name": "LCG m = 2^25 - 39", "a": 25907312, "m": 2**25 - 39},
    {"name": "LCG m = 2^29 - 3", "a": 520332806, "m": 2**29 - 3},
    {"name": "LCG m = 2^32 - 5 (alt)", "a": 279470273, "m": 2**32 - 5},
    {"name": "LCG m = 2^33 - 9", "a": 123456789, "m": 2**33 - 9},
    {"name": "LCG m = 2^40 - 87", "a": 987654321, "m": 2**40 - 87},
    {"name": "LCG m = 2^41 - 3", "a": 192837465, "m": 2**41 - 3},
    {"name": "LCG m = 2^42 - 17", "a": 918273645, "m": 2**42 - 17},
    {"name": "LCG m = 2^43 - 11", "a": 564738291, "m": 2**43 - 11},
    {"name": "LCG m = 2^44 - 27", "a": 837261945, "m": 2**44 - 27},

    # ---------------------------------------------------------
    # 5. SMALL-MODULUS TOY LCGs (fun for spectral contrast)
    # ---------------------------------------------------------
    {"name": "Toy LCG (m = 251)", "a": 33, "m": 251},
    {"name": "Toy LCG (m = 509)", "a": 273, "m": 509},
    {"name": "Toy LCG (m = 997)", "a": 123, "m": 997},
    {"name": "Toy LCG (m = 1009)", "a": 271, "m": 1009},
    {"name": "Toy LCG (m = 1499)", "a": 337, "m": 1499},
    {"name": "Toy LCG (m = 2003)", "a": 777, "m": 2003},
    {"name": "Toy LCG (m = 3001)", "a": 901, "m": 3001},
    {"name": "Toy LCG (m = 4001)", "a": 1231, "m": 4001},

    # ---------------------------------------------------------
    # 6. LARGE-MODULUS / 64-BIT FAMILIES
    # ---------------------------------------------------------
    {"name": "LCG 64-bit (a = 2862933555777941757)", "a": 2862933555777941757, "m": 2**64},
    {"name": "LCG 64-bit (a = 1442695040888963407)", "a": 1442695040888963407, "m": 2**64},
    {"name": "LCG 64-bit (a = 1181783497276652981)", "a": 1181783497276652981, "m": 2**64},
    {"name": "LCG 64-bit (a = 3935559000370003845)", "a": 3935559000370003845, "m": 2**64},
    {"name": "LCG 64-bit (a = 6364136223846793005)", "a": 6364136223846793005, "m": 2**64},
    {"name": "LCG 64-bit (a = 3202034522624059733)", "a": 3202034522624059733, "m": 2**64},
    {"name": "LCG 64-bit (a = 2806196910506780709)", "a": 2806196910506780709, "m": 2**64},

    # ---------------------------------------------------------
    # 7. CRYPTO-ADJACENT / STRANGE CHOICES
    # ---------------------------------------------------------
    {"name": "LCG with Mersenne Prime Modulus", "a": 48271, "m": 2**31 - 1},
    {"name": "LCG with Sophie Germain Prime Modulus", "a": 69621, "m": 2**29 - 3},
    {"name": "LCG with Safe Prime Modulus", "a": 123457, "m": 2**61 - 1},
    {"name": "LCG with Large Prime Modulus", "a": 912345, "m": 2**59 - 1},
    {"name": "LCG with Cryptographic Prime Modulus", "a": 712345, "m": 2**53 - 1},

]


LCG_DATABASE

results = batch_spectral_test()

df = pd.DataFrame(
    results,
    columns=["Generator", "Scores (s=2..7)"]
)

df

In [10]:
def batch_spectral_test():
    results = []

    for g in LCG_DATABASE:
        name = g["name"]
        a = g["a"]
        m = g["m"]

        scores = []

        for s in range(2, 8):  # dimensions 2..7
            B = Matrix(build_lcg_dual_basis(a, m, s))
            B_reduced = lll_reduce(B)
            shortest = find_shortest_vector(B_reduced)
            score = tezuka_spectral_score(shortest, m, s)
            scores.append(score)

        results.append((name, scores))

    return results